In [ ]:
# Pseudobulk profiles per cluster and correlation against the bulk data.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
!pwd

In [ ]:
adata.X = adata.layers['data']

In [ ]:
pseudo_bulk_profiles = adata.to_df().groupby(adata.obs['leiden_res_0_8']).mean()

In [ ]:
pseudo_bulk_profiles = pseudo_bulk_profiles.T
# Save as a .tsv file with integer values
pseudo_bulk_profiles.to_csv("/home/yanjun2/scanpy/decon/nomr_pbulk_matrix.tsv", sep='\t', quoting=3)

In [ ]:

bulk_data = pd.read_csv('/home/yanjun2/scanpy/decon/mdix.csv', index_col=0)
bulk_data

In [ ]:
# Grouping by 'leiden_res_0_8' and taking the mean
pseudo_bulk_profiles = adata.to_df().copy()  # Ensure to copy the DataFrame
pseudo_bulk_profiles['leiden_res_0_8'] = adata.obs['leiden_res_0_8']  # Add the leiden_res_0_8 column
pseudo_bulk_profiles = pseudo_bulk_profiles.groupby('leiden_res_0_8').mean()  # Group by 'leiden_res_0_8' and calculate the mean

# Check the result to confirm 'leiden_res_0_8' is now the index
print(pseudo_bulk_profiles.head())


In [ ]:
common_genes = bulk_data.index.intersection(pseudo_bulk_profiles.columns)
pseudo_bulk_profiles = pseudo_bulk_profiles[common_genes]
bulk_data = bulk_data.loc[common_genes]

In [ ]:
from scipy.stats import spearmanr

correlation_results = {}

# Loop through each cluster in pseudo_bulk_profiles
for cluster, cluster_profile in pseudo_bulk_profiles.iterrows():
    # Calculate Spearman correlation with the bulk_data (mean across samples)
    correlation, _ = spearmanr(cluster_profile, bulk_data.mean(axis=1))  
    correlation_results[cluster] = correlation

# Create a DataFrame with the results
correlation_df = pd.DataFrame.from_dict(correlation_results, orient='index', columns=['Spearman Correlation'])
correlation_df = correlation_df.sort_values(by='Spearman Correlation', ascending=False)

print(correlation_df)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
correlation_df['Spearman Correlation'].plot(kind='bar', color='skyblue')
plt.title('Spearman Correlation Between Pseudo-Bulk Profiles and Bulk Data')
plt.xlabel('Cluster')
plt.ylabel('Spearman Correlation')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.savefig('spearman.png')


In [ ]:
from scipy.stats import pearsonr

correlation_results = {}

# Loop through each cluster in pseudo_bulk_profiles
for cluster, cluster_profile in pseudo_bulk_profiles.iterrows():
    # Calculate Pearson correlation with the bulk_data (mean across samples)
    correlation, _ = pearsonr(cluster_profile, bulk_data.mean(axis=1))  
    correlation_results[cluster] = correlation

# Create a DataFrame with the results
correlation_df = pd.DataFrame.from_dict(correlation_results, orient='index', columns=['Pearson Correlation'])
correlation_df = correlation_df.sort_values(by='Pearson Correlation', ascending=False)

print(correlation_df)



In [ ]:
import matplotlib.pyplot as plt

# Create a bar plot for the Pearson correlation values
plt.figure(figsize=(10, 6))
correlation_df['Pearson Correlation'].plot(kind='bar', color='skyblue')

# Add title and labels
plt.title('Pearson Correlation Between Pseudo-Bulk Profiles and Bulk Data')
plt.xlabel('Cluster')
plt.ylabel('Pearson Correlation')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust layout for tight display
plt.tight_layout()

# Show the plot
plt.show()

plt.savefig('pearson.png')


In [ ]:
from scipy.stats import pearsonr

In [ ]:
import matplotlib.pyplot as plt

# Create the plot
plt.figure(figsize=(10, 6))
plt.bar(correlation_df.index, correlation_df['Correlation'], color='skyblue')

# Add labels and title
plt.xlabel('Cluster')
plt.ylabel('Spearman Correlation')
plt.title('Spearman Correlation between Pseudo-Bulk Profiles and Bulk Data')

# Rotate the x-axis labels for readability
plt.xticks(rotation=90)

# Show the plot
plt.tight_layout()
plt.show()
